# 2. Practicing with Meshes

## Constructing a Tetrahedron

In [1]:
import pytorch3d
import pytorch3d.structures
import torch
import matplotlib.pyplot as plt
import pytorch3d.renderer
import numpy

In [2]:

# Tetrahedron vertices and edges

vertices = torch.Tensor(
    [[0, 0, 1], [-0.5, 0, -0.5], [0.5, 0, -0.5],  [0, 1, 0.5]]
)
# define faces in counterclockwise dir when from the face outwards
faces = torch.Tensor([
    [0, 1, 2], [0, 3, 1], [0, 2, 3], [3, 2, 1]
])

textures = pytorch3d.renderer.TexturesVertex((torch.ones_like(vertices) * torch.Tensor([0.3, 1.0, 0.3]) ).unsqueeze(0))

mesh = pytorch3d.structures.Meshes(verts=torch.unsqueeze(vertices, 0), faces=torch.unsqueeze(faces, 0), textures=textures)


In [3]:
# Define renderer and light
from starter.utils import get_mesh_renderer
light = pytorch3d.renderer.DirectionalLights()


In [4]:
import imageio
frames = []
for azimuth in range(0, 360, 5):
    # camera and rotation/translation matrices
    
    R, T = pytorch3d.renderer.look_at_view_transform(dist=3, elev=-10, azim=azimuth)
    
    camera = pytorch3d.renderer.FoVPerspectiveCameras(
        R=R, T=T, fov=70
    )
    
    # render and show
    
    renderer = pytorch3d.renderer.MeshRenderer(
        rasterizer = pytorch3d.renderer.MeshRasterizer(
            cameras=camera,
            raster_settings=pytorch3d.renderer.RasterizationSettings(
                image_size=512
            ),
        ),
        shader=pytorch3d.renderer.HardFlatShader(
        cameras=camera,
        lights=light
        )
    )
    
    rend = renderer(mesh, cameras=camera, lights=light)
    
    frames.append(  (rend[0, ..., :3].numpy() * 255).astype(numpy.uint8) )
    
imageio.mimwrite("gif_tetrahedron.gif", frames, duration=1000//10, loop=0)



In [5]:
vertices.shape[0]

4

In [6]:
faces.shape[0]

4

## Constructing a Cube


In [7]:
cube_verts = torch.Tensor(
    [
        [0.5, 0, 0.5],
        [-0.5, 0, 0.5],
        [-0.5, 0, -0.5],
        [0.5, 0, -0.5],
        [0.5, 1, 0.5],
        [-0.5, 1, 0.5],
        [-0.5, 1, -0.5],
        [0.5, 1, -0.5],
    ]
)

cube_faces = torch.Tensor(
    [
        [2, 1, 0],
        [0, 3, 2],
        [4, 0, 1],
        [1, 5, 4],
        [3, 0, 4],
        [4, 7, 3],
        [6, 2, 3],
        [3, 7, 6],
        [6, 5, 1],
        [1, 2, 6]
    ]
)

texture = pytorch3d.renderer.TexturesVertex((torch.ones_like(cube_verts) * torch.Tensor([0.9, 0, 0])).unsqueeze(0) )

In [8]:
mesh = pytorch3d.structures.Meshes(
    verts=cube_verts.unsqueeze(0), faces=cube_faces.unsqueeze(0), textures=texture
)

In [9]:
frames = []

for azimuth in range(0, 360, 5):
    R, T = pytorch3d.renderer.look_at_view_transform(dist=5, elev=0, azim=azimuth)
    cameras = pytorch3d.renderer.FoVPerspectiveCameras(R=R, T=T, fov=70)
    lights = pytorch3d.renderer.DirectionalLights(direction=[[-0.5, -0.5, 0]])
    renderer = pytorch3d.renderer.MeshRenderer(
        rasterizer=pytorch3d.renderer.MeshRasterizer(
            cameras,
            pytorch3d.renderer.RasterizationSettings(image_size=500)
        ),
        shader=pytorch3d.renderer.HardFlatShader(cameras=cameras, lights=lights),
    )
    rend = renderer(mesh, cameras=cameras, lights=lights)
    frames.append( (rend[0, ..., :3].numpy() * 255).astype(numpy.uint8) )
    

In [10]:
import imageio
imageio.mimwrite( "gif_cube.gif", frames, duration=1000//25, loop=0)

In [13]:
cube_verts.shape[0]

8

In [14]:
cube_faces.shape[0]

10